# PyDI Data Integration Workflow: Videogames

This notebook demonstrates comprehensive data integration using PyDI. We'll work with vidoegame datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Data Loading and Profiling](#part-1-data-loading-and-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 46,580 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

In [1]:
from utils import get_repo_root

ROOT = get_repo_root()
INPUT_DIR = ROOT / "usecases" / "input" / "games"
OUTPUT_DIR = ROOT / "usecases" / "output" / "games"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Part 1: Data Loading and Profiling

In [2]:
from PyDI.io import load_xml

# Load DBpedia dataset
dbpedia = load_xml(
    INPUT_DIR / "data" / "dbpedia.xml",
    name="dbpedia",
    nested_handling="aggregate"
)

# Load Actors dataset
metacritic = load_xml(
    INPUT_DIR / "data" / "metacritic.xml",
    name="metacritic",
    nested_handling="aggregate"
)

# Load Golden Globes dataset
sales = load_xml(
    INPUT_DIR / "data" / "sales.xml",
    name="sales",
    nested_handling="aggregate"
)

# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 74,951


In [3]:
from PyDI.profiling import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

dbpedia:
  Rows: 46,580
  Columns: 6
  Total nulls: 25,233
  Null percentage: 9.0%
  Null counts per column:
    releaseYear: 1,409 (3.0%)
    developer: 1,221 (2.6%)
    platform: 410 (0.9%)
    series: 22,193 (47.6%)

metacritic:
  Rows: 20,494
  Columns: 8
  Total nulls: 3,718
  Null percentage: 2.3%
  Null counts per column:
    developer: 19 (0.1%)
    criticScore: 10 (0.0%)
    userScore: 1,413 (6.9%)
    ESRB: 2,276 (11.1%)

sales:
  Rows: 7,877
  Columns: 10
  Total nulls: 1,053
  Null percentage: 1.3%
  Null counts per column:
    publisher: 1 (0.0%)
    userScore: 1,052 (13.4%)



{'rows': 7877,
 'columns': 10,
 'nulls_total': 1053,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'releaseYear': 0,
  'developer': 0,
  'publisher': 1,
  'platform': 0,
  'criticScore': 0,
  'userScore': 1052,
  'ESRB': 0,
  'globalSales': 0},
 'dtypes': {'id': 'object',
  'name': 'object',
  'releaseYear': 'object',
  'developer': 'object',
  'publisher': 'object',
  'platform': 'object',
  'criticScore': 'object',
  'userScore': 'object',
  'ESRB': 'object',
  'globalSales': 'object'}}

### Attribute Coverage Analysis

In [4]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,metacritic_count,metacritic_pct,metacritic_coverage,metacritic_samples,sales_count,sales_pct,sales_coverage,sales_samples,avg_coverage,max_coverage,datasets_with_attribute
0,ESRB,0/0,0%,0.000000,N/A,18218/20494,88.9%,0.888943,"['M', 'M', 'T']",7877/7877,100.0%,1.000000,"['E', 'E', 'E']",0.629648,1.000000,2
1,criticScore,0/0,0%,0.000000,N/A,20484/20494,100.0%,0.999512,"['97.0', '98.0', '98.0']",7877/7877,100.0%,1.000000,"['76', '82', '80']",0.666504,1.000000,2
2,developer,45359/46580,97.4%,0.973787,"['Handheld Games', 'Ocean Software', 'Key (com...",20475/20494,99.9%,0.999073,"['Rockstar Games', 'Rockstar North', 'Namco']",7877/7877,100.0%,1.000000,"['Nintendo', 'Nintendo', 'Nintendo']",0.990953,1.000000,3
3,globalSales,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7877/7877,100.0%,1.000000,"['82', '35', '32']",0.333333,1.000000,1
4,id,46580/46580,100.0%,1.000000,"['dbpedia_1', 'dbpedia_2', 'dbpedia_3']",20494/20494,100.0%,1.000000,"['metacritic_1', 'metacritic_2', 'metacritic_3']",7877/7877,100.0%,1.000000,"['sales_1', 'sales_2', 'sales_3']",1.000000,1.000000,3
5,name,46580/46580,100.0%,1.000000,"['San Francisco Rush 2049', 'RoboCop (1988 vid...",20494/20494,100.0%,1.000000,"['Red Dead Redemption 2', 'Grand Theft Auto IV...",7877/7877,100.0%,1.000000,"['Wii Sports', 'Mario Kart Wii', 'Wii Sports R...",1.000000,1.000000,3
6,platform,46170/46580,99.1%,0.991198,"['Game Boy Color', 'Arcade video game', 'PlayS...",20494/20494,100.0%,1.000000,"['Xbox One', 'Xbox 360', 'Dreamcast']",7877/7877,100.0%,1.000000,"['Wii', 'Wii', 'Wii']",0.997066,1.000000,3
7,publisher,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7876/7877,100.0%,0.999873,"['Nintendo', 'Nintendo', 'Nintendo']",0.333291,0.999873,1
8,releaseYear,45171/46580,97.0%,0.969751,"['2006-01-01', '1989-01-01', '2016-01-01']",20494/20494,100.0%,1.000000,"['2018-01-01', '2008-01-01', '1999-01-01']",7877/7877,100.0%,1.000000,"['2006-01-01', '2008-01-01', '2009-01-01']",0.989917,1.000000,3
9,series,24387/46580,52.4%,0.523551,"['Rush (video game series)', 'List of RoboCop ...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.174517,0.523551,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']


### Detailed Data Profiling

In [5]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling DBpedia...


/Users/luca/PycharmProjects/PyDI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 173.61it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/games/dataset-profiles/dbpedia_profile.html
Profiling Metacritic...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 153.56it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/games/dataset-profiles/metacritic_profile.html
Profiling Sales...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 92.84it/s]

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/games/dataset-profiles/sales_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/luca/PycharmProjects/PyDI/usecases/output/games/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • dbpedia_profile.html
  • metacritic_profile.html
  • sales_profile.html


## Part 2: Entity Matching

### Step 1: Blocking

In [6]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker

# Add title_prefix directly to the original dataframes
dbpedia['name_prefix'] = dbpedia['name'].astype(str).apply(lambda x: ''.join([word[:2].upper() for word in x.split()[:3]]))
metacritic['name_prefix'] = metacritic['name'].astype(str).apply(lambda x: ''.join([word[:2].upper() for word in x.split()[:3]]))
sales['name_prefix'] = sales['name'].astype(str).apply(lambda x: ''.join([word[:2].upper() for word in x.split()[:3]]))

standard_blocker_m2d = StandardBlocker(
    metacritic, dbpedia,
    on=['name_prefix'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_m2d = standard_blocker_m2d.materialize()

sn_blocker_m2d = SortedNeighbourhoodBlocker(
    metacritic, dbpedia,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_m2d = sn_blocker_m2d.materialize()

embedding_blocker_m2d = EmbeddingBlocker(
    metacritic, dbpedia,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_m2d = embedding_blocker_m2d.materialize()



[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 9065 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 8042 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 3777 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 67074 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugR

### Step 2: Evaluate Blocking Against Ground Truth

In [10]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 1 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 2 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 3 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 8 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 9 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 11 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 14 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 21 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 25 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 27 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 32 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 35 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 39 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 49 true matches
[INFO ] root - P

{'pair_completeness': 0.883495145631068,
 'pair_quality': 0.000758317535051353,
 'reduction_ratio': 0.9997485833279943,
 'total_candidates': 240005,
 'total_possible_pairs': 954610520,
 'true_positives_found': 182,
 'total_true_pairs': 206,
 'batches_processed': 241,
 'evaluation_timestamp': '2025-10-29T10:25:59.207774',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_detailed_results.csv']}

In [11]:
standard_blocker_m2s = StandardBlocker(
    metacritic, sales,
    on=['name_prefix'],  # Block on first 3 characters of name
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_m2s = standard_blocker_m2s.materialize()

sn_blocker_m2s = SortedNeighbourhoodBlocker(
    metacritic, sales,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_m2s = sn_blocker_m2s.materialize()

token_blocker_m2s = TokenBlocker(
    metacritic, sales,
    column='name',      # Tokenize names
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
    ngram_size=5,
    ngram_type='character'
)
token_candidates_m2s = token_blocker_m2s.materialize()

embedding_blocker_m2s = EmbeddingBlocker(
    metacritic, sales,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_m2s = embedding_blocker_m2s.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 9065 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 3824 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 3503 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 28371 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugR

Now let's evaluate which blocking method we want to use for each dataset combination:

In [13]:
# Evaluate all blocking methods for both dataset combinations

evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
m2d_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2d, standard_blocker_m2d],
    'SortedNeighbourhoodBlocker': [sn_candidates_m2d, sn_blocker_m2d],
    'EmbeddingBlocking': [embedding_candidates_m2d, embedding_blocker_m2d]
}

m2s_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2s, standard_blocker_m2s],
    'SortedNeighbourhood': [sn_candidates_m2s, sn_blocker_m2s],
    'TokenBlocking': [token_candidates_m2s, token_blocker_m2s],
    'EmbeddingBlocking': [embedding_candidates_m2s, embedding_blocker_m2s]
}

# Load correspondences for evaluation
m2d_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv",
    name="m2d_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

m2s_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv",
    name="m2s_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Evaluate blocking for a2g datasets
m2d_results = []
for method_name, candidates in m2d_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2d_correspondences,candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2d'
    m2d_results.append(result)

# Evaluate blocking for m2s datasets
m2s_results = []
for method_name, candidates in m2s_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2s_correspondences,candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2s'
    m2s_results.append(result)

# Select best method for each dataset (highest pair_completeness, then highest reduction_ratio)
m2d_best = max(m2d_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
m2s_best = max(m2s_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))

print(f"Best blocking for m2d: {m2d_best['method']} (PC: {m2d_best['pair_completeness']:.3f}, RR: {m2d_best['reduction_ratio']:.3f})")
print(f"Best blocking for m2s: {m2s_best['method']} (PC: {m2s_best['pair_completeness']:.3f}, RR: {m2s_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.883
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 182/206
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.874
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 180/206
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.883
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 182/206
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.810
[INFO ] root -   Pair Quality:      0.002
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 124/153
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.830
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.999
[INFO ] ro

Best blocking for m2d: StandardBlocking (PC: 0.883, RR: 1.000)
Best blocking for m2s: EmbeddingBlocking (PC: 0.882, RR: 0.997)


### Step 3: Entity Matching with Comparators

In [39]:
from PyDI.entitymatching import StringComparator, DateComparator

# Create comparators for different attributes
comparators = [
    # Name similarity - most important for games
    StringComparator(
        column='name',
        similarity_function='jaccard',  # Good for game names
        preprocess=str.lower  # Case normalization
    ),
    
    # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
        preprocess=str.lower
    ),

    # Date proximity - games from same year likely same game
    DateComparator(
        column='releaseYear'
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [58]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=metacritic,
    df_right=dbpedia, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 46580 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 46580 elements after 0:00:0.234; 240005 blocked pairs (reduction ratio: 0.9997485833279943)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:228.360; found 6804 correspondences.


In [59]:
correspondences_m2s = matcher.match(
    df_left=metacritic,
    df_right=sales, 
    candidates=standard_blocker_m2s, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 7877 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 7877 elements after 0:00:0.083; 71971 blocked pairs (reduction ratio: 0.9995541693113944)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:65.654; found 6683 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [74]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  172
[INFO ] root -   True Negatives:  387
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 34
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.933
[INFO ] root -   Precision: 0.966
[INFO ] root -   Recall:    0.835
[INFO ] root -   F1-Score:  0.896


{'precision': 0.9662921348314607,
 'recall': 0.8349514563106796,
 'f1': 0.8958333333333334,
 'accuracy': 0.9332220367278798,
 'true_positives': 172,
 'false_positives': 6,
 'false_negatives': 34,
 'true_negatives': 387,
 'threshold_used': 0.0,
 'total_correspondences': 6804,
 'filtered_correspondences': 6804,
 'evaluation_timestamp': '2025-10-29T12:25:26.521602',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_detailed_results.csv']}

In [75]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] root - Cluster Size Distribution of 4214 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	2948	|	69.96%
[INFO ] root - 		3	|	730	|	17.32%
[INFO ] root - 		4	|	273	|	6.48%
[INFO ] root - 		5	|	125	|	2.97%
[INFO ] root - 		6	|	52	|	1.23%
[INFO ] root - 		7	|	33	|	0.78%
[INFO ] root - 		8	|	14	|	0.33%
[INFO ] root - 		9	|	14	|	0.33%
[INFO ] root - 		10	|	8	|	0.19%
[INFO ] root - 		11	|	7	|	0.17%
[INFO ] root - 		12	|	2	|	0.05%
[INFO ] root - 		13	|	1	|	0.02%
[INFO ] root - 		14	|	1	|	0.02%
[INFO ] root - 		15	|	2	|	0.05%
[INFO ] root - 		17	|	1	|	0.02%
[INFO ] root - 		22	|	1	|	0.02%
[INFO ] root - 		26	|	1	|	0.02%
[INFO ] root - 		49	|	1	|	0.02%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2948,69.957285
1,3,730,17.323208
2,4,273,6.478405
3,5,125,2.966303
4,6,52,1.233982
5,7,33,0.783104
6,8,14,0.332226
7,9,14,0.332226
8,10,8,0.189843
9,11,7,0.166113


In [76]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 4214 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2d,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=mbm_correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 6804 -> 6804 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 6804 -> 4268 
[INFO ] root - MaximumBipartiteMatching: 6804 -> 4268 correspondences
[INFO ] root - MaximumBipartiteMatching: 10936 -> 8536 entities
[INFO ] root - Cluster Size Distribution of 4268 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	4268	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/cluster_size_distribution.csv
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  18
[INFO ] root -   True Negatives:  447
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 135
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.775
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.118
[INFO ] root -   F1-Score:  0.211


In [ ]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2s = clusterer.cluster(correspondences_m2s)

# use Stable Matching to refine results to 1:1 matches
clusterer = StableMatching()
sm_correspondences_m2s = clusterer.cluster(correspondences_m2s)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2s,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=mbm_correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=sm_correspondences_m2s,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=sm_correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

correspondences_m2s = sm_correspondences_m2s

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  122
[INFO ] root -   True Negatives:  424
[INFO ] root -   False Positives: 23
[INFO ] root -   False Negatives: 31
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.910
[INFO ] root -   Precision: 0.841
[INFO ] root -   Recall:    0.797
[INFO ] root -   F1-Score:  0.819
[INFO ] root - Cluster Size Distribution of 6438 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	6338	|	98.45%
[INFO ] root - 		3	|	34	|	0.53%
[INFO ] root - 		4	|	59	|	0.92%
[INFO ] root - 		5	|	2	|	0.03%
[INFO ] root - 		6	|	4	|	0.06%
[INFO ] root - 		10	|	1	|	0.02%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/cluster_size_distribution.csv
[INFO ] root - Filtered correspondences: 6683 -> 6683 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 6683 

## Part 3: Data Fusion

In [69]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 13,314


## Step 1: Define Fusion Strategy 

In [70]:
from PyDI.fusion import DataFusionStrategy, longest_string, union, prefer_higher_trust, voting, numeric_tolerance_match

strategy = DataFusionStrategy('game_fusion_strategy')
['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', longest_string)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', voting)
strategy.add_attribute_fuser('userScore', voting)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'platform' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'developer' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'releaseYear' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'ESRB' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'criticScore' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'userScore' using rule 'voting'


## Step 2: Run Fusion

In [71]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'game_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 21389 of 21389 unique IDs
[INFO ] PyDI.fusion.engine - Created 61719 record groups from 13314 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 61719 groups:
[INFO ] PyDI.fusion.engine -     Group Size | Frequency
[INFO ] PyDI.fusion.engine -     ----------------------
[INFO ] PyDI.fusion.engine -           1 |   53562
[INFO ] PyDI.fusion.engine -           2 |    5139
[INFO ] PyDI.fusion.engine -           3 |    2035
[INFO ] PyDI.fusion.engine -           4 |     541
[INFO ] PyDI.fusion.engine -           5 |     229
[INFO ] PyDI.fusion.engine -           6 |      82
[INFO ] PyDI

Fused rows: 8,157


,_id,_fusion_group_id,_fusion_sources,id,ESRB,userScore,platform,metacritic_id,name_prefix,developer,releaseYear,globalSales,publisher,criticScore,name,_fusion_confidence,_fusion_metadata,series
0,metacritic_18234,group_0,"[metacritic, sales]",metacritic_18234,E,None,PS2,metacritic_18234,NHFA20,989 Sports,2002-01-01,0,Sony Computer Entertainment,55.0,NHL FaceOff 2003,0.636364,"{'id_rule': 'first_non_null', 'id_inputs': [{'...",NaN
1,sales_6586,group_1,"[metacritic, sales, dbpedia]",sales_6586,E,None,Wii,metacritic_14338,OC,Compile Heart,2007-01-01,0,Midas Interactive Entertainment,66,Octomania,0.555556,"{'id_rule': 'first_non_null', 'id_inputs': [{'...",None
2,dbpedia_55995,group_2,"[metacritic, dbpedia]",dbpedia_55995,T,7.5,Xbox One,metacritic_6045,DUOFTH,Amplitude Studios,2021-01-01,NaN,NaN,78.0,Dungeon of the Endless,0.700000,"{'id_rule': 'first_non_null', 'id_inputs': [{'...",None
3,metacritic_20108,group_3,"[metacritic, sales]",metacritic_20108,T,6.2,Playstation Portable,metacritic_20108,LEOFTH,Neko Entertainment,2007-01-01,0,Game Factory,38.0,Legend of the Dragon,0.727273,"{'id_rule': 'first_non_null', 'id_inputs': [{'...",NaN
4,metacritic_18078,group_4,"[metacritic, sales]",metacritic_18078,T,6.5,Playstation Portable,metacritic_18078,EARE,Team Fusion,2006-01-01,0,Electronic Arts,56.0,EA Replay,0.727273,"{'id_rule': 'first_non_null', 'id_inputs': [{'...",NaN


## Step 3: Evaluate Data Fusion

In [72]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("director_name", tokenized_match)
strategy.add_evaluation_function("actors_actor_name", tokenized_match)
strategy.add_evaluation_function("date", year_only_match)
strategy.add_evaluation_function("oscar", boolean_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'title'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'director_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'actors_actor_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'oscar'


In [73]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.723 overall accuracy (86/119)


Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.723
  macro_accuracy: 0.718
  num_evaluated_records: 15
  num_evaluated_attributes: 8
  total_evaluations: 119
  total_correct: 86
  platform_accuracy: 1.000
  platform_count: 15
  criticScore_accuracy: 0.467
  criticScore_count: 15
  developer_accuracy: 0.533
  developer_count: 15
  releaseYear_accuracy: 0.933
  releaseYear_count: 15
  name_accuracy: 0.867
  name_count: 15
  publisher_accuracy: 0.867
  publisher_count: 15
  ESRB_accuracy: 0.867
  ESRB_count: 15
  userScore_accuracy: 0.214
  userScore_count: 14

Overall Accuracy: 72.3%
